<table align="center">
  <!-- ASSTIA link -->
  <td align="center">
    <a target="_blank" href="https://trilobit-coder.github.io/deeplearning/">
      <img src="../docs/assets/logo.png" width="110px" height="70px" style="padding-bottom:5px;" />
      Visit SSTIA DeepLearning
    </a>
  </td>

  <!-- Google Colab link -->
  <td align="center">
    <a target="_blank" href="https://colab.research.google.com/github/trilobit-coder/deeplearning/blob/main/model/MNIST.ipynb">
      <img src="https://i.ibb.co/2P3SLwK/colab.png" width="110px" height="70px" style="padding-bottom:5px;" />
      Run in Google Colab
    </a>
  </td>
  
  <!-- GitHub source link -->
  <td align="center">
    <a target="_blank" href="https://github.com/trilobit-coder/deeplearning/blob/main/model/MNIST.ipynb">
      <img src="https://i.ibb.co/xfJbPmL/github.png" width="70px" height="70px" style="padding-bottom:5px;" />
      View Source on GitHub
    </a>
  </td>
</table>

## MNIST with CNN

Runing this code in colab. To get faster calculation speed, click "Runtime" at the top bar and choose "Change runtime type"-"T4 GPU"

## 10 CNN model in Pytorch

### 10.1 Dependencies

In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor

from torch.utils.data import DataLoader

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch


### 10.2 Data Preparation

In [ ]:
train_data = datasets.MNIST(
    root="data", train=True, transform=ToTensor(), download=True
)

test_data = datasets.MNIST(
    root="data", train=False, transform=ToTensor(), download=True
)

loaders = {
    "train": DataLoader(train_data, batch_size=100, shuffle=True, num_workers=1),
    "test": DataLoader(test_data, batch_size=100, shuffle=True, num_workers=1),
}

### 10.3 CNN Model

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)

        return F.softmax(x, dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_fn = nn.CrossEntropyLoss()


### 10.4 Training and Test Loop

In [ ]:

def train(epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(loaders["train"]):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 20 == 0:
            print(
                f"Train Epoch: {epoch} [{batch_idx*len(data)}/{len(loaders['train'].dataset)} ({100.0 * batch_idx/ len(loaders['train']):.0f}%)]\t{loss.item():.6f}"
            )


def test():
    model.eval()

    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in loaders["test"]:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_fn(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(loaders["test"].dataset)
    print(
        f"\nTest set: Average loss: {test_loss:.4f}, Accuracy {correct}/{len(loaders['test'].dataset)} ({100.0 * correct / len(loaders['test'].dataset):.0f}%)\n"
    )


for epoch in range(1, 6):
    train(epoch)
    test()

model.eval()


### 10.5 Test The Model With Your HandWritten Numbers

Ignore this part in your local machine

In [ ]:
import google.colab.output
from IPython.display import HTML, display
import base64
from PIL import Image
import io
import numpy as np

canvas_html = """
<div style="text-align: center;">
  <h3>Draw a digit (0-9)</h3>
  <canvas id="canvas" width="280" height="280" style="border:2px solid #000; background: black; cursor: crosshair;"></canvas>
  <br><br>
  <button id="predict_btn" style="padding: 10px 20px;">Predict</button>
  <button id="clear_btn" style="padding: 10px 20px;">Clear</button>
  <h1 id="result" style="color: blue;"></h1>
</div>

<script>
  var canvas = document.getElementById('canvas');
  var ctx = canvas.getContext('2d');
  ctx.strokeStyle = 'white';
  ctx.lineWidth = 15;
  ctx.lineCap = 'round';
  var drawing = false;

  canvas.onmousedown = (e) => { drawing = true; ctx.beginPath(); ctx.moveTo(e.offsetX, e.offsetY); };
  canvas.onmousemove = (e) => { if (drawing) { ctx.lineTo(e.offsetX, e.offsetY); ctx.stroke(); } };
  canvas.onmouseup = () => { drawing = false; };

  document.getElementById('clear_btn').onclick = () => {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    document.getElementById('result').innerHTML = '';
  };
  
  ctx.fillStyle = 'black';
  ctx.fillRect(0, 0, canvas.width, canvas.height);

  document.getElementById('predict_btn').onclick = () => {
    var dataURL = canvas.toDataURL('image/png');
    google.colab.kernel.invokeFunction('notebook.PredictDigit', [dataURL], {});
  };
</script>
"""


def predict_digit(img_data):
    header, encoded = img_data.split(",", 1)
    data = base64.b64decode(encoded)
    img = Image.open(io.BytesIO(data)).convert("L").resize((28, 28))
    img_np = np.array(img).astype(np.float32) / 255.0
    img_np = (img_np - 0.1307) / 0.3081
    img_tensor = torch.from_numpy(img_np).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        digit = output.argmax(dim=1).item()
        print(f"\rPredicted Digit: {digit} ", end="")


google.colab.output.register_callback("notebook.PredictDigit", predict_digit)
display(HTML(canvas_html))